# 🩺 Fine-tuning médical LoRA (QLoRA) — TechCorp
**Filière IA — Mission Expérimentale (Challenge IA 7h, Ynov).**

Fine-tuning **LoRA** d'un modèle de base sur le **dataset médical nettoyé par la filière DATA**
(`ruslanmv/ai-medical-chatbot`, mappé `Patient→instruction` / `Doctor→output`).

**Livrables attendus :** ce notebook (lien Colab) + les **métriques d'entraînement** (loss, epochs/steps).

> ⚠️ Modèle **expérimental**, **pas pour la production**. Ne remplace jamais un avis médical.

⚠️ **Runtime GPU obligatoire** : *Exécution → Modifier le type d'exécution → T4 GPU*.

## 0. Environnement & configuration

In [ ]:
!nvidia-smi

In [ ]:
!pip -q install "transformers>=4.44,<4.47" "peft>=0.13.0" "accelerate>=0.34.0" bitsandbytes datasets pyarrow matplotlib

In [ ]:
# --- Configuration (modifiable) ---
BASE_MODEL = "microsoft/Phi-3-mini-4k-instruct"   # cf. medical_project/Readme.md (Phi-3.x recommandé)
N_TRAIN   = 2000     # nb d'exemples médicaux pour l'entraînement POC
MAX_STEPS = 250      # borne le temps d'entraînement sur T4 (~15-20 min)
MAX_SEQ   = 512      # longueur max de séquence
SEED      = 42

## 1. Données — dataset médical nettoyé (pipeline DATA)

Reprise de la logique de `rendu/data/preparation_medical.py` : téléchargement du dataset,
nettoyage (encodage, HTML, redirections publicitaires), filtres qualité, déduplication.

In [ ]:
import os, re, json, random, unicodedata, urllib.request
import pandas as pd

URL = "https://huggingface.co/datasets/ruslanmv/ai-medical-chatbot/resolve/main/dialogues.parquet"
if not os.path.exists("dialogues.parquet"):
    print("Téléchargement du dataset médical (~142 Mo)...")
    urllib.request.urlretrieve(URL, "dialogues.parquet")
df = pd.read_parquet("dialogues.parquet", columns=["Patient", "Doctor"])
print("Dialogues bruts :", len(df))

# --- Nettoyage (identique à la filière DATA) ---
HTML   = re.compile(r"<[^>]+>")
REDIR  = re.compile(r"\s*(for (further|more) information\s*)?consult[^.]*?-+>+\s*$", re.I)
ARROW  = re.compile(r"\s*-+>+\s*")
BAD    = re.compile("[\ufffd\xa0\u200b]")

def clean(t):
    t = str(t)
    t = BAD.sub(" ", t)
    t = HTML.sub(" ", t)
    t = REDIR.sub("", t)
    t = ARROW.sub(" ", t)
    t = unicodedata.normalize("NFC", t)
    t = re.sub(r"[ \t]+", " ", t)
    return t.strip()

df["instruction"] = df["Patient"].map(clean)
df["output"]      = df["Doctor"].map(clean)
df = df[(df.instruction.str.len() >= 15) & (df.output.str.len() >= 25)]
df = df[(df.instruction.str.len() + df.output.str.len()) <= 2000]
df = df.drop_duplicates(["instruction", "output"])

records = df[["instruction", "output"]].to_dict("records")
random.seed(SEED); random.shuffle(records)
train_records = records[:N_TRAIN]
print(f"Propre : {len(records)}  ->  entraînement POC : {len(train_records)}")
print(train_records[0])

## 2. Modèle de base + LoRA (QLoRA 4-bit)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

tok = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "right"

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb, device_map="auto", trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
    target_modules=['qkv_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

## 3. Tokenisation

In [ ]:
from datasets import Dataset

def to_text(r):
    return f"<|user|>\n{r['instruction']}<|end|>\n<|assistant|>\n{r['output']}<|end|>"

ds = Dataset.from_list([{"text": to_text(r)} for r in train_records])

def tok_fn(batch):
    return tok(batch["text"], truncation=True, max_length=MAX_SEQ)

ds = ds.map(tok_fn, batched=True, remove_columns=["text"])
print(ds)

## 4. Entraînement + métriques

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

args = TrainingArguments(
    output_dir="medical_lora",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=20,
    max_steps=MAX_STEPS,
    logging_steps=10,
    save_strategy="no",
    fp16=True,
    report_to="none",
)
collator = DataCollatorForLanguageModeling(tok, mlm=False)
trainer = Trainer(model=model, args=args, train_dataset=ds, data_collator=collator)
trainer.train()

In [ ]:
# Courbe de loss (métrique demandée)
import matplotlib.pyplot as plt
hist = trainer.state.log_history
steps = [h["step"] for h in hist if "loss" in h]
loss  = [h["loss"] for h in hist if "loss" in h]
plt.figure(figsize=(7,4))
plt.plot(steps, loss, marker="o")
plt.xlabel("step"); plt.ylabel("training loss")
plt.title("Fine-tuning médical LoRA — training loss")
plt.grid(True); plt.show()

print(f"Loss initiale : {loss[0]:.4f}" if loss else "n/a")
print(f"Loss finale   : {loss[-1]:.4f}" if loss else "n/a")
print(f"Steps         : {steps[-1] if steps else 0}  |  exemples : {len(train_records)}")

## 5. Sauvegarde de l'adaptateur

In [ ]:
model.save_pretrained("medical_lora_adapter")
tok.save_pretrained('medical_lora_adapter')
print("Adaptateur LoRA médical sauvegardé dans ./medical_lora_adapter")
# Optionnel : monter Google Drive pour conserver l'adaptateur
# from google.colab import drive; drive.mount('/content/drive')
# !cp -r medical_lora_adapter /content/drive/MyDrive/

## 6. Test qualitatif (après fine-tuning)

In [ ]:
model.eval()
@torch.no_grad()
def ask_med(m, max_new_tokens=160):
    p = f'<|user|>\n{m}<|end|>\n<|assistant|>\n'
    i = tok(p, return_tensors='pt').to(model.device)
    o = model.generate(**i, max_new_tokens=max_new_tokens, do_sample=False,
                       repetition_penalty=1.1, pad_token_id=tok.eos_token_id,
                       eos_token_id=tok.eos_token_id)
    return tok.decode(o[0][i['input_ids'].shape[1]:], skip_special_tokens=True).strip()

tests = [
    'I have had a headache and mild fever for two days. What should I do?',
    'What are common causes of lower back pain?',
    'Is it safe to take ibuprofen if I have high blood pressure?',
]
for q in tests:
    print('Q:', q)
    print('R:', ask_med(q))
    print('-' * 90)

## 7. Conclusion & livrables

- **Modèle :** base Phi-3-mini + adaptateur **LoRA médical** (QLoRA 4-bit).
- **Données :** dataset médical nettoyé par la filière DATA (mapping instruction/output).
- **Métriques :** voir la courbe de loss et les valeurs (loss initiale/finale, steps) en section 4.

**Pour livrer le lien Colab :** *Fichier → Enregistrer une copie dans Drive*, puis *Partager* le
lien (accès lecture). Ajouter ce lien + une capture de la courbe de loss au rendu Moodle.

> ⚠️ **Modèle expérimental, non destiné à la production.** Toute réponse médicale doit être
> validée par un professionnel de santé. Voir `medical_project/Readme.md` (RGPD, anonymisation,
> validation clinique).